# TR 2d

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elma16/beamax/blob/main/examples/reconstruction/time-reversal/TR-2d.ipynb)

> Select **Runtime → Change runtime type → GPU or TPU** in Colab to demonstrate the hardware-acceleration story.


In [ ]:
# Install beamax and its dependencies for Google Colab.
# Safe to skip when running locally from a checkout.
%%capture
%pip install --quiet "beamax[kwave] @ git+https://github.com/elma16/beamax.git"


In [ ]:
%load_ext autoreload
%autoreload 2

import jax.numpy as jnp
from beamax import utils, geometry, transforms, plotter
from beamax.decomposition import DyadicDecomposition
from beamax.gb import gb_solvers
import jax
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from einops import rearrange
from time import time
import numpy as np


from pathlib import Path
from beamax.solvers import KWaveSolver, MSGBSolver
from beamax.solvers import hybrid_solver_utils
from kwave.options.simulation_execution_options import SimulationExecutionOptions
from kwave.options.simulation_options import SimulationOptions

jax.config.update("jax_enable_x64", True)
ROOT_DIR = utils.detect_root()
DATA_DIR = Path(ROOT_DIR / "data")
PLOT_DIR = Path(ROOT_DIR / "plots")
DATA_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

"""
This example shows the application of the forward and time reversal solvers for the linear wave equation.

Step 1: Forward simulation
- Set up the domain, sensors, and initial pressure field.
- Run the forward simulation to obtain the sensor data.

Step 2: Time reversal
- Set up a new domain with the sensor data as the initial pressure field.
- Run the time reversal simulation to obtain the time-reversed pressure field.

try and match the parameters
"""

pltgb = plotter.PlotHelper()

In [ ]:
N = (64, 64)
d = len(N)
dx = (1e-4,) * d
box_aspect_ratio = (1,) * d
num_levels = 2
num_boxes_levels = (4, 8)  # This is a fixed value for the example
windowing = "rectangular_mirror"
redundancy = 2
total_coeffs = jnp.prod(redundancy * jnp.array(N))
strategy = "top_n"
# num_beams = int(total_coeffs * 0.1)
num_beams = 200
batch_size = 100
input_type = "spatial"
output_type = "spatial"
thr_strat = "top_n"
sum_method = "scan_real"

print(f"num_GB_img_space: {num_beams}, batch_size: {batch_size}")

cfl = (jnp.sqrt(2) / 4).round(3)

periodic = (False,) * d
solver = gb_solvers.solve_ODE_base
solverODE_batch = gb_solvers.solve_ODE_batch_t


def c(x):
    return 1 + 0 * x[..., 0]
    # return 100 + jnp.exp(-((x[..., 0] - 0.5*N[0]*dx[0]) ** 2 + (x[..., 1] - 0.5*N[1]*dx[1]) ** 2) / (0.1 ** 2)) * 10


domain_img = geometry.Domain(N=N, dx=dx, c=c, cfl=cfl, periodic=periodic)
XY = domain_img.grid

x = jnp.linspace(0, 1, N[0])
y = jnp.linspace(0, 1, N[1])
X, Y = jnp.meshgrid(x, y, indexing="ij")
ts_img = domain_img.generate_time_domain()
tmax_img = ts_img[-1]
Nt = len(ts_img)
if Nt != 4 * N[0]:
    ts_img = jnp.linspace(0, tmax_img, 4 * N[0])
    tmax_img = ts_img[-1]
    Nt = len(ts_img)
    dt_img = ts_img[1] - ts_img[0]
    cfl = c(jnp.zeros(d)) * dt_img / min(dx)
    domain_img = geometry.Domain(N=N, dx=dx, c=c, cfl=cfl, periodic=periodic)


dyadic_decomp_img = DyadicDecomposition(
    num_levels, N, num_boxes_levels, box_aspect_ratio
)

# pltgb.plot_centers(dyadic_decomp.centres_ndim)

wpt_img = transforms.MSWPT(dyadic_decomp_img, redundancy, windowing)


#######################################
### SENSORS ###########################
#######################################
binary_mask = jnp.zeros(N)
binary_mask = binary_mask.at[0, ...].set(1)
# binary_mask = binary_mask.at[-1, ...].set(1)
# binary_mask = binary_mask.at[:, 0].set(1)
# binary_mask = binary_mask.at[:, -1].set(1)
sensors = geometry.Sensor(domain=domain_img, binary_mask=binary_mask)
sensors_all = geometry.Sensor(domain=domain_img, binary_mask=jnp.ones(N))

In [ ]:
from beamax import transforms

KXY = dyadic_decomp_img.fourier_meshgrid

# pltgb.plot_centers(dyadic_decomp.centres_ndim)

boxhf = 44
boxlf = 10

khf = jnp.array([19, 14])
klf = jnp.array([10, 3])
kerft_hf = transforms.compute_frames(
    dyadic_decomp_img, boxhf, khf, KXY, redundancy, "none"
)
kerft_lf = transforms.compute_frames(
    dyadic_decomp_img, boxlf, klf, KXY, redundancy, "none"
)
p0 = utils.unitary_ifft(kerft_hf) + utils.unitary_ifft(kerft_lf)
p0 = p0 / jnp.max(jnp.abs(p0))
p0 = p0.T
exp = 1


# boxhf = 63  # flat
# # boxhf = 54
# khf = jnp.array([8, 8])
# kerft_hf = transforms.compute_frames(
#     dyadic_decomp_img, boxhf, khf, KXY, redundancy, "none"
# )
# p0 = utils.unitary_ifft(kerft_hf)
# p0 = p0 / jnp.max(jnp.abs(p0))
# # p0 = p0.T
# exp = 1

#######################################
### POINT SOURCE ######################
#######################################

# p0 = jnp.zeros(N)
# p0 = p0.at[N[0] // 4, N[1] // 2].set(1)
# # # p0 = p0.at[N[1]//8::N[1]//8,N[0] // 2].set(1.0)
# # p0 = p0.at[N[0] // 8 :: N[0] // 8, N[1] // 8 :: N[1] // 8].set(1.0)
# exp = 2

######################################
# VESSELS PHANTOM ####################
######################################

# from scipy import io as sio

# data = sio.loadmat(DATA_DIR / "vessels_BB.mat")["p0"]
# p0 = data / jnp.max(jnp.abs(data))
# print(f"Shape of p0: {p0.shape}")
# exp = 3

#######################################
##  PALM VESSELS ######################
#######################################

# from scipy import io as sio

# data = sio.loadmat(DATA_DIR / "palm5.mat")["p0"][:256, :256]
# p0 = data / jnp.max(jnp.abs(data))
# exp = 5

######################################
### CIRCLES PHANTOM ##################
######################################
# from kwave.utils.mapgen import make_disc
# from kwave.data import Vector

# expected_scale = (128, 128)
# # # create initial pressure distribution using makeDisc
# N_vec = Vector(N)  # [grid points]
# disc_magnitude = 1  # [Pa]
# disc_pos = Vector([25 // 2, 60 // 2])  # [grid points]
# disc_radius = 5  # [grid points]
# disc_2 = disc_magnitude * make_disc(N_vec, disc_pos, disc_radius)

# disc_pos = Vector([80 // 2, 50 // 2])  # [grid points]
# disc_radius = 20  # [grid points]
# disc_magnitude = 2  # [Pa]
# disc_1 = disc_magnitude * make_disc(N_vec, disc_pos, disc_radius)

# p0 = disc_1 + disc_2
# exp = 6

p0 = p0.real

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(p0, origin="lower")
plt.scatter(jnp.where(sensors.binary_mask)[1], jnp.where(sensors.binary_mask)[0], c="r")
plt.colorbar()
plt.title("Initial Pressure Field")
plt.subplot(1, 2, 2)
plt.imshow(c(XY), origin="lower")
plt.colorbar()
plt.title("Sound Speed")
plt.savefig(
    PLOT_DIR / f"2d-initial-pressure-field-{exp}.png", dpi=300, bbox_inches="tight"
)
plt.show()
plt.close()

p0_fft = utils.unitary_fft(p0)

dpdt = jnp.zeros_like(p0)

plt.figure(figsize=(10, 10))
coeffs = wpt_img.convert_to_array(wpt_img.forward(p0, input_type))
centres2 = dyadic_decomp_img.centres_ndim * 2 + jnp.array(N)
plt.imshow(jnp.abs(coeffs.T), origin="lower")
plt.colorbar()
plt.scatter(centres2[:, 0], centres2[:, 1], c="r")
for idx, (x, y) in enumerate(centres2, start=0):
    plt.annotate(
        str(idx),
        (x, y),
        fontsize=10,
        ha="left",
        va="bottom",
        xytext=(5, 5),
        textcoords="offset points",
        color="white",
    )
plt.show()

## SOLVE FORWARD USING MSGB

In [ ]:
centres_img = dyadic_decomp_img.centres_ndim + jnp.array(N) // 2

msgb_solver = MSGBSolver(
    thr=num_beams,
    thr_strat=thr_strat,
    batch_size=batch_size,
    input_type=input_type,
    ode_solver=solver,
    sum_method=sum_method,
)

simulation_options = SimulationOptions(
    data_cast="double",
    smooth_p0=False,
    save_to_disk=True,
)

execution_options = SimulationExecutionOptions(
    is_gpu_simulation=False, delete_data=False, verbose_level=0, show_sim_log=False
)

kwave_solver = KWaveSolver(simulation_options, execution_options)

t1 = time()
sensor_data_kw = kwave_solver.forward(p0, domain_img, sensors.binary_mask, ts_img)
t2 = time()
print(f"k-Wave forward solve took {t2 - t1:.2f} seconds")

t1 = time()
sensor_data_gb, params_fwd = msgb_solver.forward(
    p0,
    domain_img,
    sensors,
    ts_img,
    wpt_img,
)
t2 = time()
print(f"MSGB forward solve took {t2 - t1:.2f} seconds")

# plt.imshow(sensor_data_kw, origin="lower")
# plt.colorbar()
# plt.title("k-Wave Sensor Data")
# plt.savefig(PLOT_DIR / f"2d-sensor-data-kw-{exp}.png", dpi=300, bbox_inches="tight")
# plt.show()
# plt.close()

## Time Reversal

In [ ]:
def cut_out_middle(arr, size):
    mid = arr.shape[0] // 2
    return arr[mid - size // 2 : mid + size // 2]


sensor_data_fft = utils.unitary_fft(sensor_data_kw)
sensor_data_fft_cropped = cut_out_middle(
    sensor_data_fft, 2 * jnp.sqrt(N[0] * N[1]).astype(int)
)
sensor_data_cropped = utils.unitary_ifft(sensor_data_fft_cropped)

print(f"Shape of sensor_data_fft: {sensor_data_fft.shape}")
print(f"Shape of sensor_data_cropped: {sensor_data_cropped.shape}")

energy = jnp.linalg.norm(sensor_data_fft)
cropped_energy = jnp.linalg.norm(sensor_data_cropped)
print(
    f"Energy: {energy}, Cropped Energy: {cropped_energy}, Ratio: {cropped_energy / energy}"
)


# plt.figure(figsize=(10, 5))
# plt.subplot(1, 3, 1)
# plt.imshow(jnp.log(jnp.abs(sensor_data_fft)), origin="lower")
# plt.colorbar()
# plt.title("Sensor Data FFT")
# plt.subplot(1, 3, 2)
# plt.imshow(jnp.log(jnp.abs(sensor_data_fft_cropped)), origin="lower")
# plt.colorbar()
# plt.title("Cropped Sensor Data FFT")
# plt.subplot(1, 3, 3)
# plt.imshow(sensor_data_cropped.real, origin="lower")
# plt.colorbar()
# plt.title("Cropped Sensor Data")
# plt.tight_layout()
# plt.savefig(
#     PLOT_DIR / f"2d-sensor-data-fft-cropped-{exp}.png", dpi=300, bbox_inches="tight"
# )
# plt.show()
# plt.close()


N_rect = sensor_data_cropped.shape
dpdt_rect = jnp.zeros(N_rect)

Nt = max(N_rect)
N_min = min(N_rect)
ts_data = jnp.linspace(0, tmax_img, Nt)
dt_data = float(ts_data[1] - ts_data[0])
tmax_data = ts_data[-1]

dx_rect = (dt_data,) + dx[1:]
box_aspect_ratio_rect = tuple([N_rect[i] / N_min for i in range(d)])

assert jnp.allclose(
    tmax_img, tmax_data
), f"tmax_img {tmax_img} and tmax_data {tmax_data} are not equal."
print(f"N: {N}, dx: {dx}, box_aspect_ratio: {box_aspect_ratio}")
print(
    f"N_rect: {N_rect}, dx_rect: {dx_rect}, box_aspect_ratio_rect: {box_aspect_ratio_rect}"
)
print(f"ts_img: {ts_img[-1]}, ts_data: {ts_data[-1]}")


domain_data = geometry.Domain(N=N_rect, dx=dx_rect, c=c, periodic=periodic, cfl=cfl)

dyadic_decomp_data = DyadicDecomposition(
    num_levels, N_rect, num_boxes_levels, box_aspect_ratio_rect
)

wpt_data = transforms.MSWPT(dyadic_decomp_data, redundancy, windowing)


#################################
### add noise to sensor data ####
#################################

# noise_level = 0.01
# key = jax.random.PRNGKey(0)
# sigma = noise_level * jnp.abs(sensor_data_kw)
# noise = jax.random.normal(key, sensor_data_kw.shape) * sigma
# sensor_data_kw_noisy = sensor_data_kw + noise
# sensor_data_kw = sensor_data_kw_noisy

# plt.imshow(sensor_data_kw_noisy.real, origin="lower")
# plt.colorbar()
# plt.title("Noisy Sensor Data")
# plt.savefig(PLOT_DIR / f"2d-noisy-sensor-data-{exp}.png", dpi=300, bbox_inches="tight")
# plt.show()

t1 = time()
p0_TR_msgb, params_TR = msgb_solver.time_reversal(
    sensor_data_cropped, domain_img, XY, sensors, ts_img, domain_data, wpt_data
)
t2 = time()
print(f"MSGB time reversal took {t2 - t1:.2f} seconds")

t1 = time()
p0_TR_kw = kwave_solver.time_reversal(
    sensor_data_kw.T, domain_img, jnp.ones(N), sensors.binary_mask, ts_img
).T
t2 = time()
print(f"k-Wave time reversal took {t2 - t1:.2f} seconds")

ts_0 = jnp.array([0.0])
gb_init = msgb_solver.forward(p0, domain_img, XY, ts_0, wpt_img)[0]

# plt.figure(figsize=(10, 5))
# plt.subplot(1, 4, 1)
# plt.imshow(p0, origin="lower")
# plt.colorbar()
# plt.title("Initial Pressure Field")
# plt.scatter(
#     jnp.where(binary_mask)[1],
#     jnp.where(binary_mask)[0],
#     color="red",
#     s=10,
#     label="Sensors",
# )
# plt.subplot(1, 4, 2)
# plt.imshow(jnp.real(gb_init).squeeze(), origin="lower")
# plt.scatter(
#     jnp.where(binary_mask)[1],
#     jnp.where(binary_mask)[0],
#     color="red",
#     s=10,
#     label="Sensors",
# )
# plt.colorbar()
# plt.title("GB Initial Pressure Field (MSGB)")
# plt.subplot(1, 4, 3)
# plt.imshow(jnp.real(p0_TR_msgb).squeeze(), origin="lower")
# plt.colorbar()
# plt.title("TR Pressure Field (MSGB)")
# plt.subplot(1, 4, 4)
# plt.imshow(jnp.real(p0_TR_kw).squeeze(), origin="lower")
# plt.colorbar()
# plt.title("TR Pressure Field (k-Wave)")
# plt.show()

# plt.plot(jnp.real(p0_TR_msgb)[:, N[0] // 2], label="MSGB TR")
# plt.plot(jnp.real(p0_TR_kw)[:, N[0] // 2], label="k-Wave TR")
# plt.plot(p0[:, N[0] // 2], label="Initial Pressure")
# plt.title("Pressure Field Comparison")
# plt.legend()
# plt.show()


(p0_fwd, m0_fwd, x0_fwd, ws_fwd, a0_fwd, modes_fwd) = params_fwd
(p0_tr, m0_tr, x0_tr, ws_tr, a0_tr, signum_tr, ts_tr) = params_TR


def flatten_params(params):
    return tuple(rearrange(param, "a b ... -> (a b) ...") for param in params)


p0_fwd, m0_fwd, x0_fwd, ws_fwd, a0_fwd, modes_fwd = flatten_params(
    (p0_fwd, m0_fwd, x0_fwd, ws_fwd, a0_fwd, modes_fwd)
)
p0_tr, m0_tr, x0_tr, ws_tr, a0_tr, ts_tr, signum_tr = flatten_params(
    (p0_tr, m0_tr, x0_tr, ws_tr, a0_tr, ts_tr, signum_tr)
)

# print("FWD PARAMS")
# gb_half = len(p0_fwd) // 2
# print(f"p0 fwd: {p0_fwd} p0_tr: {p0_tr * signum_tr}")
# print(f"m0 fwd: {m0_fwd} m0_tr: {m0_tr}")
# print(f"x0 fwd: {x0_fwd} x0_tr: {x0_tr}")
# print(f"ws fwd: {ws_fwd} ws_tr: {ws_tr}")
# print(f"a0 fwd: {a0_fwd} a0_tr: {a0_tr}")
# print(f"ts tr: {ts_tr}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))  # 1 row, 3 columns

# Titles help make comparisons clearer
titles = ["Original $p_0$", "Time Reversal (k-Wave)", "Time Reversal (MSGB)"]
images = [p0, p0_TR_kw, p0_TR_msgb]

for ax, img, title in zip(axes, images, titles):
    im = ax.imshow(img, aspect="auto")
    ax.set_title(title)
    ax.axis("off")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

## hybrid

In [ ]:
box_corners = jnp.array([0, 15])
# box_corners = jnp.array([16, 75])
cutoff_freq = None
downsample = False
use_pow2 = False
mask = sensors.binary_mask

# pltgb.plot_centers(dyadic_decomp_data.centres_ndim)

sensor_data_fft = utils.unitary_fft(sensor_data_kw)
sensor_data_fft_cropped = cut_out_middle(sensor_data_fft, 2 * N[0])
sensor_data_cropped = utils.unitary_ifft(sensor_data_fft_cropped)


plt.imshow(sensor_data_cropped.real, origin="lower")
plt.colorbar()
plt.title("Cropped Sensor Data")
plt.savefig(
    PLOT_DIR / f"2d-cropped-sensor-data-{exp}.png", dpi=300, bbox_inches="tight"
)
plt.show()

(
    sensor_data_HF,
    sensor_data_LF,
    ds_mask,
    ds_domain,
) = hybrid_solver_utils.split_frequency_components(
    p0=sensor_data_cropped,
    sensors_mask=mask,
    input_type="spatial",
    output_type="spatial",
    wpt=wpt_data,
    box_corners=box_corners,
    windowing=wpt_data.windowing,
    domain=domain_img,
    cutoff_freq=cutoff_freq,
    downsample=downsample,
    use_pow2=use_pow2,
)

# plt.figure(figsize=(10, 5))
# plt.imshow(
#     jnp.abs((sensor_data_cropped - (sensor_data_HF + sensor_data_LF))), origin="lower"
# )
# plt.colorbar()
# plt.title("Diff")
# plt.tight_layout()
# plt.savefig(PLOT_DIR / f"2d-diff-{exp}.png", dpi=300, bbox_inches="tight")
# plt.show()

assert jnp.allclose(
    sensor_data_cropped, sensor_data_HF + sensor_data_LF, atol=1e-6
), "HF and LF components do not sum to original data."

old_shape = sensor_data_HF.shape
new_shape = (4 * N[0], N[0])

scale_factor = jnp.sqrt(jnp.prod(jnp.array(old_shape)) / jnp.prod(jnp.array(new_shape)))

sensor_data_HF_new = (
    utils.interpolate_fourier(sensor_data_HF, new_shape, "spatial", "spatial")
    / scale_factor
)
sensor_data_LF_new = (
    utils.interpolate_fourier(sensor_data_LF, new_shape, "spatial", "spatial")
    / scale_factor
)


# alternative idea. since i needed to slice away the small bit of energy at the end,
# take the LF, pad with zeros, and the slice in the small bits from the original into the HF
# sensor_data_LF_new = sensor_data_LF

# padding_needed = 512 - 256
# padded_array = jnp.pad(sensor_data_LF, ((padding_needed, 0), (0, 0)), mode='constant', constant_values=0)
# padded_array = padded_array.at[:128,...].set(sensor_data_kw[:128,...])
# padded_array = padded_array.at[-128:,...].set(sensor_data_kw[-128:,...])


# plt.figure(figsize=(10, 5))
# plt.imshow(
#     jnp.abs((sensor_data_kw - (sensor_data_HF_new + sensor_data_LF_new))),
#     origin="lower",
# )
# plt.colorbar()
# plt.title("Diff after interpolation")
# plt.tight_layout()
# plt.savefig(
#     PLOT_DIR / f"2d-diff-after-interpolation-{exp}.png", dpi=300, bbox_inches="tight"
# )
# plt.show()


# assert jnp.allclose(sensor_data_kw, sensor_data_HF_new + sensor_data_LF_new)


# plt.figure(figsize=(10, 5))
# plt.subplot(1, 2, 1)
# plt.imshow(sensor_data_kw.real, origin="lower")
# plt.colorbar()
# plt.title("kw")
# plt.subplot(1, 2, 2)
# plt.imshow(sensor_data_LF.real, origin="lower")
# plt.colorbar()
# plt.title("lf")
# plt.tight_layout()
# plt.savefig(PLOT_DIR / f"2d-sensor-data-lf-{exp}.png", dpi=300, bbox_inches="tight")
# plt.show()


sensor_data_fft = utils.unitary_fft(sensor_data_HF_new)
sensor_data_fft_cropped = cut_out_middle(sensor_data_fft, 2 * N[0])
sensor_data_cropped = utils.unitary_ifft(sensor_data_fft_cropped)


hf_solve = msgb_solver.time_reversal(
    sensor_data_cropped, domain_data, domain_img, wpt_data, wpt_img, XY, sensors
)[0]
lf_solve = kwave_solver.time_reversal(
    sensor_data_LF_new.T,
    domain_img,
    sensors_all.binary_mask,
    sensors.binary_mask,
    ts_img,
).T

sensor_data_hybrid = hf_solve + lf_solve

print("sensor_data_HF shape", sensor_data_HF.shape)
print("sensor_data_LF shape", sensor_data_LF.shape)


##########################
### Results ##############
##########################
# print("kw p0 TR", jnp.max(jnp.abs(p0_TR_kw)))
# # print('kwave p0 TR cropped', jnp.max(jnp.abs(p0_TR_kw_cropped)))
# print("msgb p0 TR", jnp.max(jnp.abs(p0_TR_msgb)))
# # print('hybrid p0 TR', jnp.max(jnp.abs(sensor_data_hybrid)))


# # assume p0, p0_TR_msgb, p0_TR_kw, sensor_data_hybrid.real, sensors.binary_mask already exist

# # # Compute shared vmin/vmax:
# # arrays = [p0, p0_TR_msgb, p0_TR_kw, sensor_data_hybrid.real]
# # vmin = min(arr.min() for arr in arrays)
# # vmax = max(arr.max() for arr in arrays)

# # fig, axes = plt.subplots(1, 4, figsize=(10, 5))

# # # Plot each image without individual cbar and remove axes:
# # im = axes[0].imshow(p0, origin="lower", vmin=vmin, vmax=vmax)
# # axes[0].scatter(jnp.where(sensors.binary_mask)[1], jnp.where(sensors.binary_mask)[0], c="r")
# # axes[0].axis("off")
# # axes[0].set_title("Initial Pressure Field")

# # axes[1].imshow(p0_TR_msgb, origin="lower", vmin=vmin, vmax=vmax)
# # axes[1].axis("off")
# # axes[1].set_title("Time Reversed Pressure Field")

# # axes[2].imshow(p0_TR_kw, origin="lower", vmin=vmin, vmax=vmax)
# # axes[2].axis("off")
# # axes[2].set_title("k-Wave Time Reversal Result")

# # axes[3].imshow(sensor_data_hybrid.real, origin="lower", vmin=vmin, vmax=vmax)
# # axes[3].axis("off")
# # axes[3].set_title("Hybrid Time Reversal Result")

# # # Attach a colorbar to the rightmost Axes so it matches its height:
# # divider = make_axes_locatable(axes[3])
# # cax = divider.append_axes("right", size="5%", pad=0.05)
# # fig.colorbar(im, cax=cax)

# # plt.tight_layout()
# # plt.show()


# # (1) Compute shared vmin/vmax across all four arrays:
# arrays = [p0, p0_TR_msgb, p0_TR_kw, sensor_data_hybrid.real]
# vmin = min(arr.min() for arr in arrays)
# vmax = max(arr.max() for arr in arrays)

# # (2) Make a 1×4 subplot grid; figsize can be adjusted as needed:
# fig, axes = plt.subplots(1, 4, figsize=(12, 4))

# # (3) Plot each image on its own Axes, remove ticks/frame, set title:
# # Set aspect='auto' to force all images to fill their subplot areas equally
# im = axes[0].imshow(p0, origin="lower", vmin=vmin, vmax=vmax, aspect="auto")
# axes[0].scatter(
#     jnp.where(sensors.binary_mask)[1], jnp.where(sensors.binary_mask)[0], c="r"
# )
# axes[0].axis("off")
# axes[0].set_title("p0")

# axes[1].imshow(p0_TR_msgb, origin="lower", vmin=vmin, vmax=vmax, aspect="auto")
# axes[1].axis("off")
# axes[1].set_title("p0 TR (MSGB)")

# axes[2].imshow(p0_TR_kw, origin="lower", vmin=vmin, vmax=vmax, aspect="auto")
# axes[2].axis("off")
# axes[2].set_title("p0 TR (k-Wave)")

# axes[3].imshow(
#     sensor_data_hybrid.real, origin="lower", vmin=vmin, vmax=vmax, aspect="auto"
# )
# axes[3].axis("off")
# axes[3].set_title("p0 TR (Hybrid)")

# # (4) Attach a colorbar to the rightmost Axes so it exactly matches that Axes' height:
# divider = make_axes_locatable(axes[3])
# cax = divider.append_axes("right", size="5%", pad=0.05)
# fig.colorbar(im, cax=cax)

# # (5) Tweak spacing so panels butt up against each other lightly:
# plt.subplots_adjust(wspace=0.05)
# plt.savefig(PLOT_DIR / f"2d-TR-wv-{exp}.png", dpi=300)
# plt.show()


# idx = N[0]//2
# line_mask = jnp.zeros_like(p0)
# line_mask = line_mask.at[:, idx].set(1)

# for ax in axes:
#     ax.contour(line_mask, levels=[0.5], colors="w", linewidths=1)

# plt.plot(p0[:, idx], label="p0")
# plt.plot(p0_TR_msgb[:, idx], label="p0 TR (MSGB)")
# plt.plot(p0_TR_kw[:, idx], label="p0 TR (k-Wave)")
# plt.plot(sensor_data_hybrid.real[:, idx], label="p0 TR (Hybrid)")
# plt.xlabel("x")
# plt.ylabel("Pressure")
# plt.legend()
# plt.title(f"Pressure along the sensor at index {idx}")
# plt.savefig(PLOT_DIR / f"2d-TR-line-{exp}.png", dpi=300)
# plt.show()

####################################
### debugging ######################
####################################

# p0_fwd = params_fwd[0]
# m0_fwd = params_fwd[1]
# x0_fwd = params_fwd[2]
# ω_fwd = params_fwd[3]
# a0_fwd = params_fwd[4]
# modes_fwd = params_fwd[5]


# def rearrange_vars(*vars):
#     return [rearrange(var, "a b ... -> (a b) ...") for var in vars]


# p0_fwd, x0_fwd, modes_fwd, m0_fwd, ω_fwd, a0_fwd = rearrange_vars(
#     p0_fwd, x0_fwd, modes_fwd, m0_fwd, ω_fwd, a0_fwd
# )

# (xt, pt, mt, at) = solver(x0_fwd, p0_fwd, m0_fwd, a0_fwd, modes_fwd, ts, c, None)


# print("---------Params for GB in img space--------")
# print("p0_fwd", pt[:, -1])
# print("M_fwd", mt[:, -1])
# print("x0_fwd", xt[:, -1])
# print("ω_fwd", ω_fwd)
# print("a0_fwd", at[:, -1])
# print("modes_fwd", modes_fwd)


# print("--------Params for GB in data space--------")
# print("p0_bwd", params_TR[0])
# print("M_bwd", params_TR[1])
# print("x0_bwd", params_TR[2])
# print("ω_bwd", params_TR[3])
# print("a0_bwd", params_TR[4])

extent = (
    0,
    domain_img.grid_size[0],
    0,
    domain_img.grid_size[1],
)  # adjust as needed based on your domain


def tr_comparison_with_profile(
    p0,
    p0_TR_msgb,
    p0_TR_kw,
    p0_TR_hyb,
    extent,
    profile_axis="x",
    profile_pos_phys=0.015,
    sensors=None,
    cmap="RdBu_r",
):
    A = [
        p0,
        p0_TR_msgb,
        p0_TR_kw,
        (p0_TR_hyb.real if jnp.iscomplexobj(p0_TR_hyb) else p0_TR_hyb),
    ]
    vmin = min([np.min(np.asarray(a)) for a in A])
    vmax = max([np.max(np.asarray(a)) for a in A])
    fig, axes = plt.subplots(1, 4, figsize=(12, 4))
    titles = ["p0", "p0 TR (MSGB)", "p0 TR (k-Wave)", "p0 TR (Hybrid)"]
    ims = []
    for ax, arr, t in zip(axes, A, titles):
        im = ax.imshow(
            arr,
            origin="lower",
            extent=extent,
            vmin=vmin,
            vmax=vmax,
            aspect="auto",
            # cmap=cmap,
        )
        ims.append(im)
        if sensors is not None:
            rr, cc = jnp.where(sensors.binary_mask)
            xs = np.linspace(extent[0], extent[1], arr.shape[1])
            ys = np.linspace(extent[2], extent[3], arr.shape[0])
            ax.scatter(xs[cc], ys[rr], s=8, c="r")
        ax.set_title(t)
        ax.set_xticks([])
        ax.set_yticks([])
    divider = make_axes_locatable(axes[-1])
    cax = divider.append_axes("right", "5%", "5%")
    fig.colorbar(ims[0], cax=cax)
    plt.subplots_adjust(wspace=0.05)

    Ny, Nx = A[0].shape
    xs = np.linspace(extent[0], extent[1], Nx)
    ys = np.linspace(extent[2], extent[3], Ny)
    if profile_axis.lower() == "x":
        x0 = profile_pos_phys
        for ax in axes:
            ax.axvline(x0, color="k", lw=1.0, ls="--")
        idx = int(np.clip(np.round((x0 - extent[0]) / (xs[1] - xs[0])), 0, Nx - 1))
        abscissa = ys
        lines = [a[:, idx] for a in A]
        xlab = "y [m]"
        where = f"x={x0:.3g} m"
    else:
        y0 = profile_pos_phys
        for ax in axes:
            ax.axhline(y0, color="k", lw=1.0, ls="--")
        idx = int(np.clip(np.round((y0 - extent[2]) / (ys[1] - ys[0])), 0, Ny - 1))
        abscissa = xs
        lines = [a[idx, :] for a in A]
        xlab = "x [m]"
        where = f"y={y0:.3g} m"

    fig2, ax = plt.subplots(figsize=(6, 3))
    for dat, lab in zip(lines, titles):
        ax.plot(abscissa, dat, label=lab, lw=2)
    ax.set_xlabel(xlab)
    ax.set_ylabel("amplitude")
    ax.set_title(f"Profile at {where}")
    ax.legend()
    ax.grid(True)
    return fig, fig2


# same extent you use for imshow
fig, fig2 = tr_comparison_with_profile(
    p0,
    p0_TR_msgb,
    p0_TR_kw,
    sensor_data_hybrid,
    extent,
    profile_axis="x",
    profile_pos_phys=10 * dx[0],
    sensors=sensors,
)
fig.savefig(PLOT_DIR / f"2d-TR-wv-{exp}.png", dpi=300, bbox_inches="tight")
fig2.savefig(PLOT_DIR / f"2d-TR-line-{exp}.png", dpi=300, bbox_inches="tight")
plt.show()